# Cat Breed Classifier
Classify a photo of a cat into one of **12 breeds** with a convolutional neural network that is
built and trained here from **random weights**.

| | |
|---|---|
| **Task** | 12-way image classification (cat breed) |
| **Data** | Oxford-IIIT Pet, cats only: 2,371 photos (public dataset, cited below) |
| **Model** | `CatCNN`: 4 conv blocks + global average pooling, about 1.2 M parameters |
| **Metric** | Accuracy (plus macro-F1 and top-3 accuracy) on a held-out **test** set |
| **Baselines** | Majority class, and logistic regression on raw pixels |
| **Seed** | 42 everywhere |

> **Runtime → Change runtime type → T4 GPU** before you start. Then run the setup
> cell and confirm it prints `cuda`.

## 1 · Setup, GPU check & the spine

Run this once. It imports everything, fixes the seed so your runs are reproducible
(the rubric expects a fixed seed), picks the GPU, and defines the **spine** you'll
reuse no matter which track you chose:

In [ ]:
# Colab already has torch, torchvision, scikit-learn, matplotlib, pillow, tqdm. Only `datasets` is extra.
!pip install -q datasets

In [ ]:
import json, random, time, copy, warnings
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.exceptions import ConvergenceWarning

# CONFIG (the single place to change settings)
CONFIG = dict(
    seed=42,
    val_frac=0.2,          # 20 % of the official trainval split becomes validation
    img_size=128,          # every image is resized to 128 x 128
    batch_size=32,
    epochs=60,             # 60 epochs takes a few minutes per run on a T4; try 30 and img_size 96 on a CPU
    lr=2e-3,               # AdamW start learning rate (cosine decay to 0)
    weight_decay=5e-2,
    dropout=0.3,
    label_smoothing=0.1,
    width=32,              # channels of the first conv block (doubles each block)
    augment=True,
    data_root="data",      # downloaded here
    out_root="outputs",    # one folder per run: best.pt, history.json, curves.png
)
SEED = CONFIG["seed"]

# Optional: keep results when Colab disconnects.
USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    CONFIG["out_root"] = "/content/drive/MyDrive/cat_breed_classifier/outputs"
OUT_ROOT = Path(CONFIG["out_root"]); OUT_ROOT.mkdir(parents=True, exist_ok=True)


def set_seed(seed=SEED):
    """Fix every random number generator so runs reproduce."""
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


set_seed()
device = get_device()
print("device:", device, "|", torch.cuda.get_device_name(0) if device.type == "cuda" else "no GPU")
if device.type == "cpu":
    print("WARNING: no GPU. Runtime -> Change runtime type -> T4 GPU, then run again.")
    print("On CPU use epochs=30 and img_size=96 in CONFIG.")
print("torch", torch.__version__)

## 2 · Dataset and splits

**Dataset:** [Oxford-IIIT Pet](https://www.robots.ox.ac.uk/~vgg/data/pets/) (CC BY-SA 4.0), restricted to its
12 cat breeds: Abyssinian, Bengal, Birman, Bombay, British Shorthair, Egyptian Mau, Maine Coon, Persian,
Ragdoll, Russian Blue, Siamese, Sphynx.

The images are downloaded from the Hugging Face mirror
[`timm/oxford-iiit-pet`](https://huggingface.co/datasets/timm/oxford-iiit-pet) into `data/`.

| Split | Source | Used for |
|---|---|---|
| **train** | 80 % of the official `trainval`, stratified by breed, seed 42 | fitting the weights |
| **val** | the other 20 % of `trainval` | choosing the model, checkpoint and hyper-parameters |
| **test** | the official `test` split | **final evaluation only** (section 9) |


In [ ]:
CAT_BREEDS = [
    "Abyssinian", "Bengal", "Birman", "Bombay", "British Shorthair", "Egyptian Mau",
    "Maine Coon", "Persian", "Ragdoll", "Russian Blue", "Siamese", "Sphynx",
]
N_CLASSES = len(CAT_BREEDS)
HF_DATASET, FOLDER = "timm/oxford-iiit-pet", "oxford-iiit-pet"
RESAMPLE = getattr(Image, "Resampling", Image).BICUBIC


def download_from_hf(base: Path):
    """Write the cat images + two annotation files (trainval, test) from Hugging Face into `base`."""
    from datasets import Image as HFImage, load_dataset
    print(f"Downloading cats from Hugging Face ({HF_DATASET}) ...")
    (base / "images").mkdir(parents=True, exist_ok=True)
    (base / "annotations").mkdir(parents=True, exist_ok=True)
    for hf_split, out_split in (("train", "trainval"), ("test", "test")):
        ds = load_dataset(HF_DATASET, split=hf_split)
        ds = ds.cast_column("image", HFImage(decode=False))    # keep the original JPEG bytes
        lines = []
        for row in ds:
            if row["label_cat_dog"] != 0:                      # 0 = cat, 1 = dog, keep cats only
                continue
            name = row["image_id"]
            dest = base / "images" / f"{name}.jpg"
            if not dest.exists():
                dest.write_bytes(row["image"]["bytes"])
            lines.append(f"{name} {row['label'] + 1} 1 1")
        (base / "annotations" / f"{out_split}.txt").write_text("\n".join(lines) + "\n")
        print(f"  {out_split}: {len(lines)} cat images")


def parse_split(base: Path, split: str):
    """[(image_id, label 0..11)] for the cats listed in annotations/<split>.txt."""
    items = []
    for line in (base / "annotations" / f"{split}.txt").read_text().splitlines():
        if not line.strip() or line.startswith("#"):
            continue
        name, _class_id, species, _breed_id = line.split()[:4]
        if int(species) != 1:                                     # 1 = cat, 2 = dog
            continue
        breed = name.rsplit("_", 1)[0].replace("_", " ")          # Ex. change "Maine_Coon_12" to "Maine Coon"
        if breed not in CAT_BREEDS:
            raise ValueError(f"Unexpected breed '{breed}' in '{name}'. Is the dataset corrupted?")
        items.append((name, CAT_BREEDS.index(breed)))
    return items


def ensure_dataset(root="data", download=True) -> Path:
    base = Path(root) / FOLDER

    def complete():
        for split in ("trainval", "test"):
            if not (base / "annotations" / f"{split}.txt").exists():
                return False
            if any(not (base / "images" / f"{n}.jpg").exists() for n, _ in parse_split(base, split)):
                return False
        return True

    if not complete():
        if not download:
            raise FileNotFoundError(f"Dataset not found in '{base}'.")
        download_from_hf(base)
        if not complete():
            raise RuntimeError(f"Dataset in '{base}' is still incomplete after downloading.")
    return base


def get_splits(root="data", val_frac=0.2, seed=42, download=True):
    """(train, val, test): lists of (image_path, label), in a deterministic order."""
    base = ensure_dataset(root, download)

    def read(split):
        return sorted((str(base / "images" / f"{n}.jpg"), y) for n, y in parse_split(base, split))

    trainval, test = read("trainval"), read("test")
    idx_train, idx_val = train_test_split(
        np.arange(len(trainval)), test_size=val_frac, random_state=seed,
        stratify=[y for _, y in trainval])
    train = [trainval[i] for i in sorted(idx_train)]
    val = [trainval[i] for i in sorted(idx_val)]

    # Leakage check: no image may appear in two splits.
    p_train, p_val, p_test = ({p for p, _ in s} for s in (train, val, test))
    assert not (p_train & p_val) and not (p_train & p_test) and not (p_val & p_test), "DATA LEAKAGE!"
    return train, val, test


def class_counts(items):
    return np.bincount([y for _, y in items], minlength=N_CLASSES)


def resize_square(img: Image.Image, size: int) -> Image.Image:
    """The ONE resize used for training, evaluation and prediction (so there is no train/test skew)."""
    return img.convert("RGB").resize((size, size), RESAMPLE)


def load_images(items, size: int) -> np.ndarray:
    """Decode + resize every image once -> uint8 array (N, size, size, 3)."""
    X = np.empty((len(items), size, size, 3), dtype=np.uint8)
    for i, (path, _) in enumerate(items):
        with Image.open(path) as im:
            X[i] = np.asarray(resize_square(im, size))
    return X


train_items, val_items, test_items = get_splits(
    CONFIG["data_root"], CONFIG["val_frac"], SEED, download=True)

print(f"train={len(train_items)}  val={len(val_items)}  test={len(test_items)}  (no image shared between splits)")
ct, cv, cs = class_counts(train_items), class_counts(val_items), class_counts(test_items)
print(f"\n{'breed':20s} {'train':>6s} {'val':>5s} {'test':>5s}")
for i, breed in enumerate(CAT_BREEDS):
    print(f"{breed:20s} {ct[i]:6d} {cv[i]:5d} {cs[i]:5d}")

## 3 · Look at the data

Seeing the data is where you notice class balance and look-alike breeds.

In [ ]:
# class balance of the three splits
x = np.arange(N_CLASSES); w = 0.28
fig, ax = plt.subplots(figsize=(11, 3.6))
ax.bar(x - w, ct, w, label="train"); ax.bar(x, cv, w, label="val"); ax.bar(x + w, cs, w, label="test")
ax.set_xticks(x); ax.set_xticklabels(CAT_BREEDS, rotation=45, ha="right")
ax.set_ylabel("images"); ax.set_title("Images per breed in each split"); ax.legend()
plt.tight_layout(); plt.show()

# one random TRAIN photo per breed
rng = np.random.default_rng(SEED)
fig, axes = plt.subplots(2, 6, figsize=(13, 5))
for ax, b in zip(axes.ravel(), range(N_CLASSES)):
    path = rng.choice([p for p, y in train_items if y == b])
    with Image.open(path) as im:
        ax.imshow(im.convert("RGB")); ax.set_title(f"{CAT_BREEDS[b]}\n{im.size[0]}x{im.size[1]}", fontsize=9)
    ax.axis("off")
plt.tight_layout(); plt.show()
print(f"\nmajority-class accuracy on train ~ {ct.max() / ct.sum():.1%}   (chance = {1 / N_CLASSES:.1%})")

## 4 · Baselines

1. **Majority class:** always predict the most common **train** breed.
2. **Logistic regression on raw 32 × 32 pixels:** a simple linear model. Its regularisation strength is chosen on **validation**.

In [ ]:
BASE_SIZE = 32   # 32 x 32 x 3 = 3072 features

y_tr = np.array([y for _, y in train_items]); y_va = np.array([y for _, y in val_items])
X_tr = load_images(train_items, BASE_SIZE).reshape(len(train_items), -1).astype(np.float32) / 255.0
X_va = load_images(val_items, BASE_SIZE).reshape(len(val_items), -1).astype(np.float32) / 255.0

# majority class
majority_label = int(np.bincount(y_tr).argmax())
majority_val_acc = float(np.mean(y_va == majority_label))

# logistic regression
set_seed()
scaler = StandardScaler().fit(X_tr)
Z_tr, Z_va = scaler.transform(X_tr), scaler.transform(X_va)
best_lr = None
for C in (1e-4, 1e-3, 1e-2, 1e-1):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", ConvergenceWarning)
        clf = LogisticRegression(C=C, max_iter=1000).fit(Z_tr, y_tr)
    va = float(accuracy_score(y_va, clf.predict(Z_va)))
    print(f"  logistic regression C={C:<7g} val acc {va:.3f}")
    if best_lr is None or va > best_lr["val_acc"]:
        best_lr = dict(C=C, val_acc=va, clf=clf)

baselines_val = {"majority": majority_val_acc, "logreg_pixels": best_lr["val_acc"]}
print(f"\n{'baseline':16s} {'val acc':>8s}")
print(f"{'majority':16s} {majority_val_acc:8.3f}   (always '{CAT_BREEDS[majority_label]}')")
print(f"{'logreg_pixels':16s} {best_lr['val_acc']:8.3f}   (C = {best_lr['C']:g})")

## 5 · Build your model

```
input 3 x 128 x 128
block 1   2 x [conv3x3 - BatchNorm - ReLU]    3 -> 32,  max-pool   -> 32 x 64 x 64
block 2   2 x [conv3x3 - BatchNorm - ReLU]   32 -> 64,  max-pool   -> 64 x 32 x 32
block 3   2 x [conv3x3 - BatchNorm - ReLU]   64 -> 128, max-pool   -> 128 x 16 x 16
block 4   2 x [conv3x3 - BatchNorm - ReLU]  128 -> 256, max-pool   -> 256 x 8 x 8
global average pooling -> Dropout(0.3) -> Linear(256 -> 12)  =  12 class logits
```
* Stacked 3 × 3 convolutions cover a large area with few weights.
* BatchNorm keeps training stable on a small dataset.
* Global average pooling replaces a big fully-connected layer: fewer parameters, less over-fitting.
* With only about 950 training images, regularisation matters: augmentation, dropout, weight decay, label smoothing.
* All weights start random.

In [ ]:
def conv_block(in_ch, out_ch):
    """Two (conv3x3 - BatchNorm - ReLU) layers followed by a 2x2 max-pool (halves height and width)."""
    return nn.Sequential(
        nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        nn.MaxPool2d(2),
    )


class CatCNN(nn.Module):
    def __init__(self, n_classes=12, width=32, dropout=0.3):
        super().__init__()
        w = width
        self.features = nn.Sequential(conv_block(3, w), conv_block(w, 2 * w),
                                      conv_block(2 * w, 4 * w), conv_block(4 * w, 8 * w))
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(8 * w, n_classes))

    def forward(self, x): # x: (batch, 3, H, W)
        return self.head(self.pool(self.features(x)).flatten(1))


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


# Print the shape after every block
m = CatCNN(N_CLASSES, CONFIG["width"], CONFIG["dropout"]).eval()
x = torch.zeros(1, 3, CONFIG["img_size"], CONFIG["img_size"])
print(f"input   {tuple(x.shape)}")
with torch.no_grad():
    for i, block in enumerate(m.features, 1):
        x = block(x)
        print(f"block{i}  {tuple(x.shape)}   params {count_params(block):>9,}")
print(f"total trainable parameters: {count_params(m):,}")
del m, x

## 6 · Data loaders, the training loop, and a sanity check

Augmentation is applied to the training set only.
Validation and test images are **never** augmented. The numbers are just normalisation constants.

The training loop is the five-step heartbeat: forward, loss, zero_grad, backward, step.  
Model selection looks at validation accuracy only the checkpoint of the best validation epoch is saved as `best.pt`.

In [ ]:
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)


def eval_transform():
    """Validation / test / predict: no augmentation (images are already resized)."""
    return transforms.Compose([transforms.ToTensor(), transforms.Normalize(MEAN, STD)])


def train_transform(img_size, augment=True):
    if not augment:
        return eval_transform()
    return transforms.Compose([
        transforms.RandomResizedCrop(img_size, scale=(0.6, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize(MEAN, STD),
    ])


class CatDataset(Dataset):
    """Images are decoded once into RAM (uint8 array); the transform runs per sample."""
    def __init__(self, images, labels, transform):
        self.images, self.labels, self.transform = images, [int(v) for v in labels], transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        return self.transform(Image.fromarray(self.images[i])), self.labels[i]


# Decode the TRAIN and VAL images once per image size
_IMG_CACHE = {}
def cached_images(split, items, size):
    if (split, size) not in _IMG_CACHE:
        _IMG_CACHE[(split, size)] = load_images(items, size)
    return _IMG_CACHE[(split, size)]


def make_loaders(cfg):
    """Train + validation loaders. The test images are never decoded here."""
    size = cfg["img_size"]
    Xtr, Xva = cached_images("train", train_items, size), cached_images("val", val_items, size)
    g = torch.Generator(); g.manual_seed(cfg["seed"])                           # fixed shuffling order
    train_loader = DataLoader(CatDataset(Xtr, y_tr, train_transform(size, cfg["augment"])),
                              batch_size=cfg["batch_size"], shuffle=True, drop_last=True, generator=g)
    val_loader = DataLoader(CatDataset(Xva, y_va, eval_transform()), batch_size=cfg["batch_size"])
    return train_loader, val_loader


@torch.no_grad()
def evaluate(model, loader, loss_fn, bar=None):
    """One no-grad pass over `loader`; returns (mean loss, accuracy)."""
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        total_loss += loss_fn(logits, y).item() * len(y)
        correct += (logits.argmax(dim=-1) == y).sum().item()
        n += len(y)
        if bar is not None:
            bar.update(1)
    return total_loss / n, correct / n


def train_epoch(model, loader, loss_fn, optimizer, bar=None):
    """One pass over the training set. Returns (loss, accuracy) measured while training."""
    model.train()
    total_loss, correct, n = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)                          # 1. forward
        loss = loss_fn(logits, y)                  # 2. loss
        optimizer.zero_grad()                      # 3. clear the previous gradients
        loss.backward()                            # 4. backprop
        optimizer.step()                           # 5. update the weights
        total_loss += loss.item() * len(y)
        correct += (logits.argmax(dim=-1) == y).sum().item()
        n += len(y)
        if bar is not None:
            bar.update(1)
    return total_loss / n, correct / n


def plot_curves(history, best_epoch=None, title="", path=None, show=True):
    """Train vs. validation loss and accuracy - read the gap between the two lines."""
    epochs = range(1, len(history["train_loss"]) + 1)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.8))
    ax1.plot(epochs, history["train_loss"], label="train"); ax1.plot(epochs, history["val_loss"], label="val")
    ax1.set(xlabel="epoch", ylabel="loss", title=f"Loss {title}")
    ax2.plot(epochs, history["train_acc"], label="train"); ax2.plot(epochs, history["val_acc"], label="val")
    ax2.set(xlabel="epoch", ylabel="accuracy", title=f"Accuracy {title}")
    for ax in (ax1, ax2):
        ax.legend(); ax.grid(alpha=0.3)
        if best_epoch:
            ax.axvline(best_epoch, color="grey", linestyle="--", linewidth=1) # the chosen checkpoint
    plt.tight_layout()
    if path:
        plt.savefig(path, dpi=120)
    plt.show() if show else plt.close(fig)


KNOBS = ["epochs", "lr", "weight_decay", "dropout", "label_smoothing", "width", "img_size", "batch_size", "augment"]


def train_run(name, force=False, **overrides):
    """Train one run (CONFIG + overrides) and save best.pt / history.json / curves.png into outputs/<name>/.
    best.pt = the epoch with the best VALIDATION accuracy. The test set is never touched here."""
    cfg = {**CONFIG, **overrides}
    out = OUT_ROOT / name
    out.mkdir(parents=True, exist_ok=True)
    if not force and (out / "history.json").exists() and (out / "best.pt").exists():
        saved = json.loads((out / "history.json").read_text())
        if all(saved["config"].get(k) == cfg[k] for k in KNOBS):
            print(f"[{name}] already trained with this exact config - loaded from {out}/")
            return saved

    set_seed(cfg["seed"])
    train_loader, val_loader = make_loaders(cfg)
    model = CatCNN(N_CLASSES, cfg["width"], cfg["dropout"]).to(device)
    loss_fn = nn.CrossEntropyLoss(label_smoothing=cfg["label_smoothing"])
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg["epochs"])
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    best = {"val_acc": -1.0, "epoch": 0}
    changed = {k: cfg[k] for k in KNOBS if cfg[k] != CONFIG[k]}
    print(f"[{name}] device={device}  train={len(train_loader.dataset)}  val={len(val_loader.dataset)}  "
          f"params={count_params(model):,}  changed vs CONFIG: {changed or 'nothing'}")

    t0 = time.time()
    bar = tqdm(total=cfg["epochs"] * (len(train_loader) + len(val_loader)), unit="batch")
    for epoch in range(1, cfg["epochs"] + 1):
        bar.set_description(f"{name} epoch {epoch}/{cfg['epochs']}")
        tr_loss, tr_acc = train_epoch(model, train_loader, loss_fn, optimizer, bar)
        va_loss, va_acc = evaluate(model, val_loader, loss_fn, bar)
        scheduler.step()
        for key, value in zip(history, (tr_loss, tr_acc, va_loss, va_acc)):
            history[key].append(value)
        if va_acc > best["val_acc"]: # model selection looks at VALIDATION only
            best = {"val_acc": va_acc, "epoch": epoch}
            torch.save({"model_state": model.state_dict(), "config": cfg, "classes": list(CAT_BREEDS),
                        "epoch": epoch, "val_acc": va_acc}, out / "best.pt")
        if epoch == 1 or epoch % 5 == 0 or epoch == cfg["epochs"]:
            tqdm.write(f"  epoch {epoch:3d}/{cfg['epochs']} | train loss {tr_loss:.3f} acc {tr_acc:.3f} "
                       f"| val loss {va_loss:.3f} acc {va_acc:.3f}")
    bar.close()

    result = {"config": cfg, "best_epoch": best["epoch"], "best_val_acc": best["val_acc"], "history": history,
              "seconds": round(time.time() - t0, 1)}
    (out / "history.json").write_text(json.dumps(result, indent=2))
    plot_curves(history, best["epoch"], title=f"({name})", path=out / "curves.png", show=False)
    print(f"[{name}] best validation accuracy {best['val_acc']:.3f} at epoch {best['epoch']}  "
          f"({result['seconds']:.0f}s)  ->  saved {out}/best.pt")
    return result


print("training code ready: make_loaders(), train_run(), plot_curves(), evaluate()")

### Sanity check first: can the model memorise one tiny batch?

If a fresh model cannot drive the loss to about 0 on **16 images** (no dropout, no augmentation), the bug is in
the wiring (a wrong shape, a missing `zero_grad`), not in the hyper-parameters. Fix that before any long run.
This cell uses its own throw-away model, so it does not touch your real runs.

In [ ]:
def overfit_one_batch(n_images=16, steps=80, lr=3e-3):
    set_seed()
    ds = CatDataset(cached_images("train", train_items, CONFIG["img_size"]), y_tr, eval_transform())
    idx = np.random.RandomState(SEED).choice(len(ds), n_images, replace=False)   # 16 random train images
    xb = torch.stack([ds[i][0] for i in idx]).to(device)
    yb = torch.tensor([ds[i][1] for i in idx]).to(device)
    probe = CatCNN(N_CLASSES, CONFIG["width"], dropout=0.0).to(device)
    opt = torch.optim.AdamW(probe.parameters(), lr=lr, weight_decay=0.0)
    loss_fn = nn.CrossEntropyLoss()
    probe.train()
    for step in range(steps):
        logits = probe(xb)
        loss = loss_fn(logits, yb)
        opt.zero_grad(); loss.backward(); opt.step()
        if step % 20 == 0 or step == steps - 1:
            acc = (logits.argmax(-1) == yb).float().mean().item()
            print(f"step {step:3d}: loss={loss.item():.4f}  batch accuracy={acc:.0%}")
    print("-> loss should fall towards 0 and accuracy towards 100 %. If not, check the wiring.")

overfit_one_batch()

### Main run: the baseline model `cnn`

Defaults from `CONFIG` (augmentation on, dropout 0.3, lr 2e-3, 60 epochs). Every later experiment changes **one**
thing compared with this run. Tune on **validation** only; the test set stays sealed.

In [ ]:
RUNS = {}
RUNS["cnn"] = train_run("cnn")

## 7 · Experiments: one change at a time (validation numbers only)

Each run differs from the baseline `cnn` in exactly **one** setting, so the effect can be read directly:

| Run | Change vs. baseline |
|---|---|
| `cnn_noaug` | no data augmentation |
| `cnn_dropout05` | dropout 0.5 instead of 0.3 |
| `cnn_lr1e-3` | learning rate 1e-3 instead of 2e-3 |

> The validation set has only 238 images, so differences of 2–3 points are within noise. Do not over-read them.

In [ ]:
RUNS["cnn_noaug"]     = train_run("cnn_noaug", augment=False)
RUNS["cnn_dropout05"] = train_run("cnn_dropout05", dropout=0.5)
RUNS["cnn_lr1e-3"]    = train_run("cnn_lr1e-3", lr=1e-3)

In [ ]:
from IPython.display import Markdown, display

def experiments_table(baseline="cnn"):
    base_cfg = RUNS[baseline]["config"]
    lines = ["| Run | Change vs. baseline | Best val acc | Train acc (same epoch) | Train-val gap | Best epoch |",
             "|---|---|---|---|---|---|"]
    for name, r in RUNS.items():
        diffs = [f"{k}={r['config'][k]}" for k in KNOBS if r["config"][k] != base_cfg[k]]
        change = "baseline" if name == baseline else (", ".join(diffs) or "-")
        tr_acc = r["history"]["train_acc"][r["best_epoch"] - 1]
        lines.append(f"| `{name}` | {change} | {r['best_val_acc']:.3f} | {tr_acc:.3f} | "
                     f"{tr_acc - r['best_val_acc']:+.3f} | {r['best_epoch']} |")
    return "\n".join(lines)

print(f"Baselines on VALIDATION:  majority {baselines_val['majority']:.3f}   "
      f"logistic regression {baselines_val['logreg_pixels']:.3f}\n")
display(Markdown("### Experiments (validation set)\n\n" + experiments_table()))

## 8 · Read the curves, choose the model, look at validation mistakes

**How to read the loss curves.** Close and low = good. Train falling while val rises = **over-fitting**.
Both high = **under-fitting**. The dashed grey line marks the epoch whose weights were saved as `best.pt`.
Note that the training accuracy is measured on augmented batches while learning, so it is a little pessimistic.

In [ ]:
for name, r in RUNS.items():
    plot_curves(r["history"], r["best_epoch"], title=f"({name})")

# validation accuracy and validation loss
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.8))
for name, r in RUNS.items():
    ax1.plot(r["history"]["val_acc"], label=name); ax2.plot(r["history"]["val_loss"], label=name)
ax1.set(xlabel="epoch", ylabel="val accuracy", title="Validation accuracy, all runs")
ax2.set(xlabel="epoch", ylabel="val loss", title="Validation loss, all runs")
for ax in (ax1, ax2):
    ax.grid(alpha=0.3); ax.legend()
plt.tight_layout(); plt.show()

### Choose the final model (by validation accuracy only)

The run with the best **validation** accuracy is the one evaluated on the test set, **once**, in section 9.

In [ ]:
CHOSEN = max(RUNS, key=lambda k: RUNS[k]["best_val_acc"])
print(f"chosen run: '{CHOSEN}'   best validation accuracy {RUNS[CHOSEN]['best_val_acc']:.3f} "
      f"at epoch {RUNS[CHOSEN]['best_epoch']}")


def load_run(run_dir):
    """Rebuild the model saved by train_run(). Returns (model, config, class names, checkpoint)."""
    ck = torch.load(Path(run_dir) / "best.pt", map_location=device, weights_only=True)
    cfg, classes = ck["config"], ck["classes"]
    model = CatCNN(len(classes), cfg["width"], cfg["dropout"]).to(device)
    model.load_state_dict(ck["model_state"])
    model.eval()
    return model, cfg, classes, ck

In [ ]:
# Confusion matrix on VALIDATION
def plot_confusion(cm, classes, title):
    cm_norm = cm / cm.sum(axis=1, keepdims=True).clip(min=1)
    fig, ax = plt.subplots(figsize=(8.5, 7.5))
    im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(len(classes))); ax.set_xticklabels(classes, rotation=60, ha="right")
    ax.set_yticks(range(len(classes))); ax.set_yticklabels(classes)
    ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title(title)
    for i in range(len(classes)):
        for j in range(len(classes)):
            if cm[i, j]:
                ax.text(j, i, int(cm[i, j]), ha="center", va="center", fontsize=8,
                        color="white" if cm_norm[i, j] > 0.5 else "black")
    fig.colorbar(im, fraction=0.046, pad=0.04)
    plt.tight_layout(); plt.show()


@torch.no_grad()
def predict_probs(model, loader):
    """Softmax probabilities (N, classes) and true labels (N,) for every image in `loader`."""
    model.eval()
    probs, labels = [], []
    for x, y in loader:
        probs.append(torch.softmax(model(x.to(device)), dim=1).cpu())
        labels.append(y)
    return torch.cat(probs).numpy(), torch.cat(labels).numpy()


model, cfg_chosen, classes, ck = load_run(OUT_ROOT / CHOSEN)
_, val_loader = make_loaders(cfg_chosen)
pv, yv = predict_probs(model, val_loader)
cm_val = confusion_matrix(yv, pv.argmax(1), labels=list(range(N_CLASSES)))
plot_confusion(cm_val, classes, f"Confusion matrix on VALIDATION ({CHOSEN})")
recall_val = cm_val.diagonal() / np.maximum(cm_val.sum(axis=1), 1)
for c, r in sorted(zip(classes, recall_val), key=lambda t: t[1])[:4]:
    print(f"  weakest on val: {c:<18s} recall {r:.2f}")

## 9 · ✅ Final test evaluation: run this **once**

This is the only place the **test** set is opened. The chosen model (section 8) and both baselines are
scored on it together. The number printed here is the one that goes in the report.

* The cell **refuses to run a second time** in the same session. Looking at the test score and then tuning again
  turns the test set into a second validation set and quietly inflates the result.
* If something crashes *before* the guard is set, you can simply run the cell again.
* Everything after this cell (section 10) only **reads** the stored results. Do not go back and tune.

In [ ]:
# ✅ ONE-SHOT held-out test evaluation. Do NOT tune after seeing these numbers.
if globals().get("TEST_EVALUATED", False):
    raise RuntimeError("The test set was already evaluated once in this session. Re-running it after more tuning "
                       "is test leakage. To reproduce from scratch: Runtime -> Restart session, then Run all.")

model, cfg_chosen, classes, ck = load_run(OUT_ROOT / CHOSEN)
set_seed(cfg_chosen["seed"])
size = cfg_chosen["img_size"]
y_te = np.array([y for _, y in test_items])
test_paths = [p for p, _ in test_items]

X_te = load_images(test_items, size)
test_loader = DataLoader(CatDataset(X_te, y_te, eval_transform()), batch_size=64)
probs, y_true = predict_probs(model, test_loader)
y_pred, conf = probs.argmax(axis=1), probs.max(axis=1)
n = len(y_true)

acc = float((y_pred == y_true).mean())
se = float(np.sqrt(acc * (1 - acc) / n))
cm_test = confusion_matrix(y_true, y_pred, labels=list(range(N_CLASSES)))
recall = cm_test.diagonal() / np.maximum(cm_test.sum(axis=1), 1)
confusions = sorted(((int(cm_test[i, j]), classes[i], classes[j]) for i in range(N_CLASSES)
                     for j in range(N_CLASSES) if i != j and cm_test[i, j] > 0), reverse=True)[:5]

maj_pred = np.full_like(y_true, majority_label)
X_te32 = load_images(test_items, BASE_SIZE).reshape(n, -1).astype(np.float32) / 255.0
lr_pred = best_lr["clf"].predict(scaler.transform(X_te32))

TEST = {
    "run": CHOSEN, "n_test": int(n), "test_acc": acc,
    "test_acc_95ci": [acc - 1.96 * se, acc + 1.96 * se],
    "test_macro_f1": float(f1_score(y_true, y_pred, average="macro")),
    "test_top3_acc": float(np.mean([t in row for t, row in zip(y_true, np.argsort(-probs, axis=1)[:, :3])])),
    "test_loss": float(-np.mean(np.log(probs[np.arange(n), y_true] + 1e-12))),
    "per_breed_recall": {c: float(r) for c, r in zip(classes, recall)},
    "top_confusions": [{"count": c, "true": t, "predicted": p} for c, t, p in confusions],
    "checkpoint_epoch": int(ck["epoch"]), "val_acc_of_checkpoint": float(ck["val_acc"]),
    "baselines": {
        "majority": {"predicts": CAT_BREEDS[majority_label], "val_acc": baselines_val["majority"],
                     "test_acc": float(accuracy_score(y_true, maj_pred)),
                     "test_macro_f1": float(f1_score(y_true, maj_pred, average="macro", zero_division=0))},
        "logreg_pixels": {"C": best_lr["C"], "val_acc": baselines_val["logreg_pixels"],
                          "test_acc": float(accuracy_score(y_true, lr_pred)),
                          "test_macro_f1": float(f1_score(y_true, lr_pred, average="macro", zero_division=0))},
    },
}
TEST_EVALUATED = True # This cell cannot run again

(OUT_ROOT / CHOSEN / "test_metrics.json").write_text(json.dumps(TEST, indent=2))
(OUT_ROOT / "baselines.json").write_text(json.dumps(TEST["baselines"], indent=2))

lo, hi = TEST["test_acc_95ci"]
print(f"TEST accuracy {acc:.3f}  (95% interval {lo:.3f}-{hi:.3f})   n={n}")
print(f"     macro-F1 {TEST['test_macro_f1']:.3f}   top-3 accuracy {TEST['test_top3_acc']:.3f}   loss {TEST['test_loss']:.3f}")
print(f"validation accuracy of this checkpoint (epoch {ck['epoch']}): {ck['val_acc']:.3f}")
b = TEST["baselines"]
print(f"\nmajority class (always '{b['majority']['predicts']}'):  {b['majority']['test_acc']:.3f}")
print(f"logistic regression on pixels:  {b['logreg_pixels']['test_acc']:.3f}")
print(f"CatCNN ('{CHOSEN}'):             {acc:.3f}")
print("\nbeat majority baseline:           ", "YES" if acc > b["majority"]["test_acc"] else "NO")
print("beat logistic-regression baseline:", "YES" if acc > b["logreg_pixels"]["test_acc"] else "NO")
print("weakest breeds (recall): " + ", ".join(f"{classes[i]} {recall[i]:.2f}" for i in np.argsort(recall)[:3]))
print("most common confusions:  " + "; ".join(f"{t} -> {p} x{c}" for c, t, p in confusions[:3]))

In [ ]:
def make_results_md():
    """The two ready-to-paste result tables, built from the REAL numbers above."""
    b, (lo, hi) = TEST["baselines"], TEST["test_acc_95ci"]
    t2 = ["### Held-out test set (each model evaluated once)", "",
          "| Model | Test accuracy | Macro-F1 |", "|---|---|---|",
          f"| Majority class (always '{b['majority']['predicts']}') | {b['majority']['test_acc']:.3f} | {b['majority']['test_macro_f1']:.3f} |",
          f"| Logistic regression on 32x32 pixels | {b['logreg_pixels']['test_acc']:.3f} | {b['logreg_pixels']['test_macro_f1']:.3f} |",
          f"| **CatCNN `{TEST['run']}`** | **{TEST['test_acc']:.3f}** (95% {lo:.3f}-{hi:.3f}) | **{TEST['test_macro_f1']:.3f}** |"]
    return "### Experiments (validation set)\n\n" + experiments_table() + "\n\n" + "\n".join(t2) + "\n"

results_md = make_results_md()
(OUT_ROOT / "results.md").write_text(results_md)
display(Markdown(results_md))

## 10 · Failure analysis and demo on real input

A single accuracy number hides *which* breeds fail and *why*. Below: the confusion matrix on the **test** set,
per-breed recall, and the **8 most confident wrong predictions**. They are the worst mistakes, not hand-picked ones.

> From here on, nothing is tuned. These cells only read the stored test results.

In [ ]:
plot_confusion(cm_test, classes, "Confusion matrix on the TEST set (colour = row share, numbers = images)")

order = np.argsort(recall)
fig, ax = plt.subplots(figsize=(9, 3.6))
ax.bar([classes[i] for i in order], recall[order])
ax.axhline(TEST["test_acc"], color="grey", linestyle="--", label=f"overall accuracy {TEST['test_acc']:.2f}")
ax.set_ylabel("recall"); ax.set_title("Per-breed recall on the test set (worst first)")
plt.xticks(rotation=45, ha="right"); ax.legend(); plt.tight_layout(); plt.show()

print("Most common confusions (count, true -> predicted):")
for c in TEST["top_confusions"]:
    print(f"  {c['count']:3d} x  {c['true']}  ->  {c['predicted']}")

In [ ]:
wrong = np.where(y_pred != y_true)[0]
worst = wrong[np.argsort(-conf[wrong])][:8] # most CONFIDENT mistakes first
fig, axes = plt.subplots(2, 4, figsize=(15, 8))
for ax in axes.ravel():
    ax.axis("off")
for ax, i in zip(axes.ravel(), worst):
    ax.imshow(Image.open(test_paths[i]).convert("RGB"))
    ax.set_title(f"true: {classes[y_true[i]]}\npred: {classes[y_pred[i]]} ({conf[i]:.0%})", fontsize=10, color="#c62828")
fig.suptitle("Most CONFIDENT wrong predictions on the test set", fontsize=13)
plt.tight_layout(); plt.show()

### One honest failure

Look at the confusion matrix and the 8 photos above, then answer in your own words:

* **Which breeds** are confused with which?
* **What does the image look like**: pose, lighting, background, a mixed-breed cat, a very close-up face?
* **Your hypothesis for why.** Say clearly that it is a *hypothesis*; you have not tested it.
* **Is the label even certain?** Mixed cats still carry a single label in this dataset.

Look-alike pairs such as Birman vs. Ragdoll or British Shorthair vs. Russian Blue are hard even for people, but
check what *your* results say rather than assuming.

In [ ]:
# Demo on REAL input: one success and one failure from the stored test results.
def show_prediction(img, p, title, color):
    top = np.argsort(-p)[:3]
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(9, 3.2), gridspec_kw={"width_ratios": [1, 1.3]})
    a1.imshow(img); a1.axis("off"); a1.set_title(title, color=color, fontsize=11)
    a2.barh([classes[i] for i in top][::-1], p[top][::-1], color=color)
    a2.set_xlim(0, 1); a2.set_xlabel("probability"); a2.set_title("top-3 predictions")
    plt.tight_layout(); plt.show()

right = np.where(y_pred == y_true)[0]
i_ok, i_bad = right[np.argmax(conf[right])], worst[0]
show_prediction(Image.open(test_paths[i_ok]).convert("RGB"), probs[i_ok],
                f"SUCCESS: true {classes[y_true[i_ok]]}", "#2e7d32")
show_prediction(Image.open(test_paths[i_bad]).convert("RGB"), probs[i_bad],
                f"FAILURE: true {classes[y_true[i_bad]]}", "#c62828")

In [ ]:
# Try your OWN photo, run the cell and pick a file.
# The model only knows 12 breeds, so any other input is still forced into one of them.
@torch.no_grad()
def predict_image(img):
    model.eval()
    x = eval_transform()(resize_square(img.convert("RGB"), size)).unsqueeze(0).to(device)
    return torch.softmax(model(x), dim=1)[0].cpu().numpy()

try:
    from google.colab import files
    uploaded = files.upload()
except Exception:
    uploaded = {}
    print("Not running in Colab (or no file chosen): skipping the upload demo.")

for fname in uploaded:
    img = Image.open(fname).convert("RGB")
    p = predict_image(img)
    note = "  (low confidence - the model is unsure)" if p.max() < 0.5 else ""
    show_prediction(img, p, f"{fname}: {classes[p.argmax()]} {p.max():.0%}{note}", "#1565c0")